# Day 24: Decision Tree Pruning

**Topic:** Cost-Complexity Pruning - Overfitting se bachne ke liye tree ki branches kaatna

**Steps:**
1. Full tree train karo (bina constraint) — overfit hoga
2. `cost_complexity_pruning_path` se har branch ka alpha nikaalo
3. Har alpha ke liye tree train karo + accuracy check karo
4. Plot se best alpha select karo (jahan test accuracy peak ho)
5. Best alpha ke saath final pruned tree train karo
6. Tree ko visualize karo + rules print karo

## Step 1: Import Libraries

In [1]:
import seaborn as sns

## Step 2: Load Iris Dataset

150 rows, 4 features (sepal_length, sepal_width, petal_length, petal_width), 3 species (setosa, versicolor, virginica)

In [2]:
df = sns.load_dataset('iris')

## Step 3: Import sklearn modules

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

## Step 4: Train-Test Split (80-20)

80% data train ke liye, 20% test ke liye. `stratify` se har class ka ratio same rahega.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['species']), df['species'], test_size=0.2, random_state=42, stratify=df['species'])

## Step 5: Train Full Tree (Without Pruning)

Bina kisi constraint (`max_depth`, `min_samples_split`) ke tree banate hain.
Isse tree full grow karega aur overfit hone ki sambhavna hai.

In [5]:
model_full = DecisionTreeClassifier(random_state=42)
model_full.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

## Step 6: Check Full Tree Accuracy

Train accuracy 100% hogi (overfit), Test accuracy around 93%.

In [6]:
print("Train Accuracy:", model_full.score(X_train, y_train))
print("Test Accuracy:", model_full.score(X_test, y_test))

Train Accuracy: 1.0
Test Accuracy: 0.9333333333333333


## Step 7: Cost Complexity Pruning Path

`cost_complexity_pruning_path()` full tree ke har internal branch ke liye **alpha** calculate karta hai.
Alpha = accuracy_loss / complexity_reduction. Sabse chhota alpha wali branch pehle kaati jati hai.

Jitni internal branches hain, utne + 1 alphas milte hain.

Output: 6 alphas ka array (0 se 0.333 tak)

In [7]:
path = model_full.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas
ccp_alphas

array([0.        , 0.00625   , 0.00811404, 0.03392857, 0.27067669,
       0.33333333])

## Step 8: Har Alpha ke liye Tree Train + Accuracy Store

Loop har alpha ke liye naya DecisionTree banata hai, ccp_alpha set karta hai, fit karta hai,
aur train/test accuracy list me store karta hai.

In [8]:
train_scores = []
test_scores = []

for alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    clf.fit(X_train, y_train)
    train_scores.append(clf.score(X_train, y_train))
    test_scores.append(clf.score(X_test, y_test))

## Step 9: Alpha vs Accuracy Plot

**X-axis:** alpha values | **Y-axis:** Accuracy

- Blue line (Train): alpha badhne ke saath girti hai (100% → 67%)
- Orange line (Test): pehle upar jaati hai (93% → 97%), phir girne lagti hai
- **Jahan Test peak par ho, wahi best alpha**

In [9]:
import matplotlib.pyplot as plt

plt.plot(ccp_alphas, train_scores, marker='o', label='Train')
plt.plot(ccp_alphas, test_scores, marker='o', label='Test')
plt.xlabel('alpha')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

<Figure size 640x480 with 1 Axes>

## Step 10: Select Best Alpha

Jahaan test accuracy maximum hai, woh alpha best hai.
Is case me alpha = 0.00625 par test accuracy 96.7% peak par hai.

In [10]:
best_alpha = ccp_alphas[test_scores.index(max(test_scores))]
best_alpha

np.float64(0.00625)

## Step 11: Train Final Pruned Model

Best alpha (0.00625) ke saath `ccp_alpha` set karke final tree train karte hain.
Ab tree utni hi branches kaatega jitna is alpha me allow hai.

In [11]:
model_pruned = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)
model_pruned.fit(X_train, y_train)

DecisionTreeClassifier(ccp_alpha=np.float64(0.00625), random_state=42)

## Step 12: Pruned Model Accuracy Check

Comapre karo:
| Model | Train | Test |
|-------|-------|------|
| Full tree | 100% | 93.3% |
| Pruned tree | 99.2% | **96.7%** |

Test accuracy improve hui kyunki unnecessary branches kaat di gayin (overfit kam hua).

In [12]:
print("Pruned Train Accuracy:", model_pruned.score(X_train, y_train))
print("Pruned Test Accuracy:", model_pruned.score(X_test, y_test))

Pruned Train Accuracy: 0.9916666666666667
Pruned Test Accuracy: 0.9666666666666667


## Step 13: Visualize Pruned Tree

Tree ka diagram — har box me: condition, gini impurity, sample count, aur class distribution.

In [13]:
from sklearn.tree import plot_tree

plt.figure(figsize=(15, 8))
plot_tree(model_pruned, feature_names=X_train.columns, class_names=model_pruned.classes_, filled=True)
plt.show()

<Figure size 1500x800 with 1 Axes>

## Step 14: Print Decision Rules

Tree ke saare decisions text format me. Har line ek if-else condition hai.

In [14]:
from sklearn.tree import export_text

rules = export_text(model_pruned, feature_names=list(X_train.columns))
print(rules)

|--- petal_length <= 2.45
|   |--- class: setosa
|--- petal_length >  2.45
|   |--- petal_width <= 1.65
|   |   |--- petal_length <= 4.95
|   |   |   |--- class: versicolor
|   |   |--- petal_length >  4.95
|   |   |   |--- class: virginica
|   |--- petal_width >  1.65
|   |   |--- petal_length <= 4.85
|   |   |   |--- sepal_width <= 3.00
|   |   |   |   |--- class: virginica
|   |   |   |--- sepal_width >  3.00
|   |   |   |   |--- class: versicolor
|   |   |--- petal_length >  4.85
|   |   |   |--- class: virginica



## Summary

- **Full tree**: Train 100%, Test 93.3% — overfit (data ratt liya)
- **Pruned tree (alpha=0.00625)**: Train 99.2%, Test 96.7% — best balance
- **Pruning** ne unnecessary branches kaat di, model generalize better karta hai
- **petal_length** sabse important feature hai, phir **petal_width**, phir **sepal_width**. **sepal_length** ka use nahi hua